# RT-DETR UI Element Detection Training on Kaggle GPU

This notebook clones your `IrpAI` repository and fine-tunes the RT-DETR-R18 object detection model on a massive merged UI dataset (~31,000 images):
1. **ScreenSpot (All Splits):** Mobile, Desktop, and Web (2,853 samples)
2. **GroundUI-18k:** Cross-platform GUI elements (18,026 samples)
3. **Rico Mobile UI:** Android app layouts (10,000 sample subset)

### Instructions:
1. In Kaggle, go to **Add-ons** > **Secrets** (top menu bar of the notebook editor).
2. Add a new secret with Label: `GITHUB_TOKEN` and set the Value to your GitHub Personal Access Token (PAT) that has access to the `IrpAI` repository.
3. Go to **Settings** (right sidebar) and verify that **Accelerator** is set to **GPU T4 x2** or **GPU T4 x1**.
4. Toggle the **Internet on** setting (required to download the datasets and clone GitHub).
5. Click **Run All** to run the notebook.
6. Download the final zipped model checkpoint (`rtdetr-ui-complete-checkpoint-best.zip`) from the Output section.

## 1. Clone GitHub Repository

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
try:
    github_token = user_secrets.get_secret("GITHUB_TOKEN")
    repo_url = f"https://{github_token}@github.com/satgunsodhi/IrpAI.git"
    print("Cloning repository satgunsodhi/IrpAI...")
    os.system(f"git clone {repo_url} IrpAI")
except Exception as e:
    print(f"Error accessing Kaggle secrets / cloning: {e}")
    print("Please make sure you have added GITHUB_TOKEN as a Kaggle secret.")

## 2. Install Package & Dependencies

In [ ]:
%cd IrpAI
!pip install -q transformers accelerate torchvision huggingface_hub pycocotools safetensors fiftyone
!pip install -q -e ".[training]"

## 3. Prepare and Merge Datasets

In [ ]:
!python examples/training/prepare_all_datasets.py

## 4. Execute Fine-Tuning

In [ ]:
!python examples/training/finetune_rtdetr_ui.py \
    --train-images data/merged_coco/train/images \
    --train-annotations data/merged_coco/train/annotations.json \
    --val-images data/merged_coco/val/images \
    --val-annotations data/merged_coco/val/annotations.json \
    --output-dir checkpoints/rtdetr-ui-complete \
    --epochs 15 \
    --batch-size 8 \
    --lr 3e-5

## 5. Package and Export the Best Model Checkpoint

In [ ]:
!zip -r ../rtdetr-ui-complete-checkpoint-best.zip checkpoints/rtdetr-ui-complete/checkpoint-best